In [ ]:
!pip install pyspark -q

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Superstore_Final.csv to Superstore_Final (1).csv


# Week 5 — Spark Fundamentals: Data Cleaning, Transformation & Aggregation
### Dataset: Superstore_Final.csv
### Submitted by: Saksham

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week_5_Assignment") \
    .getOrCreate()

spark

In [ ]:
df = spark.read.csv("Superstore_Final.csv", header=True, inferSchema=True)

df.printSchema()
df.show(5)
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)
 |-- user_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- status: string (nullable = true)
 |-- age: string (nul

## Q1: What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?

**Answer:**

Traditional MapReduce has several limitations that Spark addresses:

1. **Disk I/O Overhead** — MapReduce writes intermediate results to disk (HDFS) after every Map and Reduce stage. For multi-stage jobs (common in ML and iterative algorithms), this means repeated disk read/write cycles, making it very slow.

2. **No In-Memory Computation** — Each MapReduce job is isolated; data cannot be cached in memory across jobs. Every new job re-reads input from disk, even if the same dataset was just processed.

3. **High Latency** — Due to constant disk I/O, MapReduce jobs have high startup and execution latency, making it unsuitable for interactive or real-time analytics.

4. **Rigid Programming Model** — MapReduce only supports Map and Reduce operations. Complex pipelines (joins, filters, multiple aggregations) require chaining multiple MapReduce jobs manually, increasing code complexity.

5. **Poor Support for Iterative Algorithms** — Machine learning algorithms (e.g., gradient descent, k-means) require multiple passes over the same data. MapReduce re-reads and re-writes data from disk on every iteration, making it extremely inefficient for ML workloads.

6. **No Built-in Support for Streaming/Interactive Queries** — MapReduce is purely batch-oriented; it cannot handle real-time streaming or ad-hoc interactive querying efficiently.

**Spark's advantages over these limitations:**
- In-memory computing (RDDs/DataFrames cached in RAM) — up to 100x faster for iterative jobs
- Rich, easy-to-use APIs (DataFrame, SQL, MLlib, GraphX, Streaming) in one unified engine
- DAG-based execution engine optimizes the entire pipeline instead of running isolated jobs
- Supports batch, streaming, ML, and graph processing in a single framework

## Q2: Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.

**Answer:**

In-Memory Computing is Spark's core performance advantage, especially critical for iterative ML algorithms (e.g., logistic regression, k-means, gradient descent) that require multiple passes over the same dataset.

**How it works:**

1. **RDD/DataFrame Caching** — Spark allows datasets to be explicitly cached in memory using `.cache()` or `.persist()`. Once cached, the data resides in the Executor JVM's memory (RAM) instead of being re-read from disk on every access.

2. **Avoiding Repeated Disk I/O** — In disk-based systems like traditional MapReduce, each iteration of an ML algorithm requires:
   - Reading the dataset from disk
   - Processing it
   - Writing intermediate results back to disk
   
   Spark eliminates this by keeping the dataset resident in memory across iterations — read once, reuse many times.

3. **DAG Scheduling with Lazy Evaluation** — Spark builds a logical execution plan (DAG) and only computes when an action is triggered. This allows Spark to optimize the entire chain of operations, minimizing redundant computation across iterations.

4. **Speed Impact** — Because RAM access is roughly 100x faster than disk access, ML algorithms that need 10-100 iterations over the same data see dramatic speedups (Spark is often cited as 10x-100x faster than Hadoop MapReduce for iterative workloads).

5. **Fault Tolerance without Replication** — Even though data sits in memory, Spark's lineage graph (DAG) ensures that if an Executor fails and loses cached data, it can be recomputed from the original source — so in-memory caching doesn't sacrifice reliability.

**Practical Example:** In k-means clustering, the same dataset is scanned in every iteration to recompute cluster centroids. With Spark's in-memory caching (`df.cache()`), this dataset is loaded once and reused across all iterations, whereas MapReduce would re-read it from HDFS every single time.

## Q3: Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.

**Answer:**

Use `.dropDuplicates()` with a subset of columns. This removes rows that have identical values in `user_id` AND `transaction_date`, keeping only one occurrence.

In [ ]:
print(f"Rows before removing duplicates: {df.count()}")

df_no_dupes = df.dropDuplicates(subset=["user_id", "transaction_date"])

print(f"Rows after removing duplicates: {df_no_dupes.count()}")
df_no_dupes.show(5)

Rows before removing duplicates: 9994
Rows after removing duplicates: 5211
+------+--------------+----------+----------+--------------+-----------+----------------+---------+-------------+------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------------------+--------+--------+------+--------+----------------+----------+---------+------------+-------------+-------------------+--------------------+----------+------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|   Customer Name|  Segment|      Country|        City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|               Sales|Quantity|Discount|Profit| user_id|transaction_date|    status|      age|subscription|raw_timestamp|              email|            username|  store_id| price|
+------+--------------+----------+----------+--------------+-----------+----------------+---------+------------

## Q4: Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.

**Answer:**

Filter rows where `Region == 'West'`, then group by `Category` (mapped from `product_category`) and compute the average of `Sales` (mapped from `sale_amount`) using `.groupBy().agg(avg(...))`.

**Note:** During execution, 300 rows in the `Sales` column contained malformed values (e.g., `" Black"`, `" Executive Red"`) caused by a CSV parsing issue where text from the `Product Name` column leaked into `Sales` due to unescaped commas/quotes. These were cleaned using `try_cast`, which safely converts invalid values to `NULL` instead of crashing the job — `NULL` values are automatically ignored by `avg()`.

In [ ]:
from pyspark.sql.functions import avg, round as spark_round, col, expr

# Clean Sales column: malformed values (data quality issue) become NULL safely
df = df.withColumn("Sales", expr("try_cast(Sales as double)"))

df_west_avg = df.filter(col("Region") == "West") \
                 .groupBy("Category") \
                 .agg(spark_round(avg("Sales"), 2).alias("avg_sale_amount"))

df_west_avg.show()

+---------------+---------------+
|       Category|avg_sale_amount|
+---------------+---------------+
|Office Supplies|         117.49|
|      Furniture|          360.6|
|     Technology|         422.64|
+---------------+---------------+



## Q5: What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.

**Answer:**

- **`.na.drop()`** — Removes rows that contain null values. By default it drops a row if ANY column has a null (`how='any'`), but can be configured with `how='all'` (drop only if ALL columns are null) or `subset=[...]` to check specific columns only. This **reduces the row count** of the DataFrame.

- **`.na.fill()`** — Replaces null values with a specified default value, WITHOUT removing any rows. You can fill different values for different columns by passing a dictionary. This **preserves the row count** while making the data complete/usable for downstream operations.

**When to use which:**
- Use `.na.drop()` when missing data makes a row meaningless (e.g., missing primary key).
- Use `.na.fill()` when you want to retain the row but need a placeholder (e.g., categorical fields like `status`, or numeric fields where 0 is a safe default).

In [ ]:
# Check null count in 'status' column before fill
null_status_count = df.filter(df["status"].isNull()).count()
print(f"Null values in 'status' column: {null_status_count}")

# Fill nulls in 'status' with 'Unknown'
df_status_filled = df.na.fill({"status": "Unknown"})

# Verify — no more nulls in status
null_status_after = df_status_filled.filter(df_status_filled["status"].isNull()).count()
print(f"Null values in 'status' after fill: {null_status_after}")

df_status_filled.select("Order ID", "status").show(10)

# For comparison: .na.drop() example (drops rows where status is null)
df_status_dropped = df.na.drop(subset=["status"])
print(f"Rows after na.drop() on 'status': {df_status_dropped.count()}  (vs original {df.count()})")

Null values in 'status' column: 778
Null values in 'status' after fill: 0
+--------------+---------+
|      Order ID|   status|
+--------------+---------+
|CA-2016-152156|Completed|
|CA-2016-152156|Cancelled|
|CA-2016-138688|  Shipped|
|US-2015-108966|  Shipped|
|US-2015-108966|Completed|
|CA-2014-115812|Completed|
|CA-2014-115812|Completed|
|CA-2014-115812|  Pending|
|CA-2014-115812|  Shipped|
|CA-2014-115812|  Shipped|
+--------------+---------+
only showing top 10 rows
Rows after na.drop() on 'status': 9216  (vs original 9994)


## Q6: Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.

**Answer:**

Group by `City`, count records per group using `.count()`, then apply a `.filter()` on the aggregated result — this is the Spark DataFrame equivalent of SQL's `HAVING` clause.

In [ ]:
city_counts = df.groupBy("City") \
                 .count() \
                 .filter("count > 100") \
                 .orderBy("count", ascending=False)

city_counts.show(20)
print(f"Number of cities with more than 100 records: {city_counts.count()}")

+-------------+-----+
|         City|count|
+-------------+-----+
|New York City|  915|
|  Los Angeles|  747|
| Philadelphia|  537|
|San Francisco|  510|
|      Seattle|  428|
|      Houston|  377|
|      Chicago|  314|
|     Columbus|  222|
|    San Diego|  170|
|  Springfield|  163|
|       Dallas|  157|
| Jacksonville|  125|
|      Detroit|  115|
+-------------+-----+

Number of cities with more than 100 records: 13


## Q7: How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?

**Answer:**

Spark DataFrames are **immutable** — once created, a DataFrame's structure and data cannot be changed in place. Every "cleaning" operation (dropping columns, renaming, filtering, casting types) does NOT modify the original DataFrame; instead, it returns a **new DataFrame** that must be captured in a new (or same) variable.

**Practical implications for data cleaning:**

1. **Must reassign the result** — Writing `df.drop("column_name")` alone has no effect on `df`. You must write `df = df.drop("column_name")` or assign to a new variable like `df_clean = df.drop("column_name")`.

2. **Enables safe chaining** — Because each transformation returns a new DataFrame, cleaning steps can be chained fluently:
```python
   df_clean = df.dropDuplicates().na.fill({"status": "Unknown"}).withColumnRenamed("old", "new")
```
   Each step in the chain produces an intermediate (lazy) DataFrame, with the final variable holding the fully cleaned result.

3. **Original data stays safe** — Since the source DataFrame is never mutated, you can always go back to the raw `df` for debugging or trying a different cleaning approach, without re-reading the file from disk.

4. **Encourages functional/pipeline style** — Cleaning steps are naturally expressed as a sequence of transformations rather than imperative in-place edits (unlike Pandas where `inplace=True` mutates the DataFrame directly).

5. **Supports Lazy Evaluation & Fault Tolerance** — Because transformations create new logical DataFrames rather than mutating data, Spark can track the full lineage (DAG) of every cleaning step, enabling automatic recovery if a partition is lost during processing.

**Example:**
```python
df_dropped = df.drop("Postal Code")          # df unchanged, df_dropped is new
df_renamed = df_dropped.withColumnRenamed("Customer Name", "CustName")  # another new DataFrame
```

## Q8: Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.

**Answer:**

Use `.between(18, 30)` for the inclusive age range condition, combined with `&` (AND) for the subscription condition.

In [ ]:
from pyspark.sql.functions import col

df_young_premium = df.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)

df_young_premium.select("Customer Name", "age", "subscription", "Sales").show(10)
print(f"Matching records: {df_young_premium.count()}")

+---------------+---+------------+-------+
|  Customer Name|age|subscription|  Sales|
+---------------+---+------------+-------+
|    Claire Gute| 27|     Premium| 261.96|
|      Pete Kriz| 19|     Premium| 665.88|
|    Emily Burns| 19|     Premium|1044.63|
|Tracy Blumstein| 26|     Premium|  9.618|
|Tracy Blumstein| 18|     Premium|  15.76|
|     Erin Smith| 27|     Premium| 95.616|
|Ted Butterfield| 24|     Premium|  48.48|
|  Karen Daniels| 27|     Premium|  75.88|
|  Paul Gonzalez| 25|     Premium|   6.16|
|       Jim Sink| 26|     Premium| 73.584|
+---------------+---+------------+-------+
only showing top 10 rows
Matching records: 1349


## Q9: When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

**Answer:**

Handling null values before aggregation is critical for several reasons:

1. **Crashes on Malformed Data** — As seen in this very assignment (Q4), the `Sales` column contained 300 malformed string values (e.g., `" Black"`, `" Executive Red"`) due to a CSV parsing issue. Running `avg()` directly on the raw column caused Spark to throw a `CAST_INVALID_INPUT` / `NumberFormatException` and crash the entire job. Cleaning the column first (via `try_cast`, converting bad values to `NULL`) prevented this.

2. **Silent Skewing of Results** — Even when nulls don't cause a crash, functions like `avg()` and `sum()` silently exclude rows with `NULL` in the aggregated column. If you don't know how many nulls exist, your average could be calculated on a much smaller, biased subset of data without you realizing it — leading to misleading business insights.

3. **Denominator Drift in avg()** — `avg()` only divides by the count of non-null values, not total row count. If 30% of a column is null, the average reported may not represent the "true" average expected by stakeholders, who often assume averages are computed across all records.

4. **Inconsistent Aggregation Behavior** — Different aggregation functions handle nulls differently: `count()` ignores nulls by default unless you use `count("*")`, `sum()` returns `NULL` if ALL values are null (not 0), and `min()/max()` simply skip nulls. Without explicit null handling, these inconsistencies can produce confusing or contradictory results across a pipeline.

5. **Downstream Pipeline Stability** — If null-laden data flows into further transformations (joins, casting, ML feature engineering), it can propagate errors or `NULL` poisoning across multiple stages, making the eventual root cause much harder to debug.

**Conclusion:** Cleaning nulls upfront (via `.na.fill()`, `.na.drop()`, or `try_cast` for malformed types) ensures aggregations are computed on a known, predictable, and crash-free dataset — exactly the pattern demonstrated in Q4 of this assignment.

## Q10: Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

**Answer:**

The `raw_timestamp` column in this dataset contains **inconsistent date formats** (e.g., `08-11-2016 00:47` in DD-MM-YYYY and `2016-11-08 00:31:55` in YYYY-MM-DD format mixed together). A direct `.cast(TimestampType())` would fail or silently produce `NULL` for the mismatched format. We handle this using `to_timestamp()` with `coalesce()` to try multiple formats safely.

In [ ]:
from pyspark.sql.functions import expr, coalesce, col

df = df.withColumn(
    "event_time",
    coalesce(
        expr("try_to_timestamp(raw_timestamp, 'yyyy-MM-dd HH:mm:ss')"),
        expr("try_to_timestamp(raw_timestamp, 'dd-MM-yyyy HH:mm')")
    )
).drop("raw_timestamp")

df.select("event_time").show(10, truncate=False)
print(f"Null event_time after conversion: {df.filter(col('event_time').isNull()).count()}")

+-------------------+
|event_time         |
+-------------------+
|2016-11-08 00:47:00|
|2016-11-08 00:31:55|
|2016-06-12 11:06:20|
|2015-10-11 02:37:20|
|2015-10-11 01:20:19|
|2014-06-09 20:23:35|
|2014-06-09 08:57:05|
|2014-06-09 02:52:00|
|2014-06-09 23:05:28|
|2014-06-09 19:20:56|
+-------------------+
only showing top 10 rows
Null event_time after conversion: 300


## Q11: Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

**Answer:**

**What is Shuffle?**

Shuffle is the process of redistributing data across partitions (and across Executor nodes) over the network, so that all records sharing the same key end up on the same partition. It is one of the most expensive operations in Spark because it involves:
- Disk I/O (writing shuffle data to disk on the map side)
- Network I/O (transferring data between Executors)
- Serialization/Deserialization of data

**Why groupBy() triggers a shuffle:**

When you run `df.groupBy("Category").agg(avg("Sales"))`, Spark must ensure all rows with the same `Category` value end up together to compute the aggregate correctly. Since data is initially spread across partitions arbitrarily (based on how it was read from the source file), Spark must:
1. Hash-partition the data by the `Category` key
2. Write intermediate results to disk on each Executor (shuffle write)
3. Transfer relevant partitions across the network to the Executor responsible for each key (shuffle read)
4. Combine and aggregate the data on the receiving Executor

**Why it's a "Wide" Transformation:**

Transformations are classified based on how data dependencies map between input and output partitions:

- **Narrow Transformation** — Each input partition contributes to exactly ONE output partition (e.g., `filter()`, `select()`, `withColumn()`). No data movement across partitions is needed; each Executor can process its partition independently.

- **Wide Transformation** — Each output partition may depend on data from MULTIPLE input partitions, requiring data to move across the cluster (e.g., `groupBy()`, `join()`, `distinct()`, `repartition()`). This data movement IS the shuffle.

Since `groupBy()` needs to collect rows with the same key from potentially every partition in the cluster, it is inherently a wide transformation — it cannot guarantee correctness without shuffling data across the network first.

**Performance Implication:** Wide transformations are significantly slower than narrow ones due to network and disk overhead, which is why minimizing unnecessary `groupBy()`/`join()` calls and using techniques like broadcast joins (for small tables) is a key Spark optimization strategy.

## Q12: Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

**Answer:**

This dataset has 499 nulls in `email` and 399 nulls in `username` (stored as `NULL`, not literal empty strings, since it's CSV-sourced). We check both conditions: `email IS NULL` and `username` being either an empty string `""` or `NULL` (to cover both possible representations of "empty").

In [ ]:
from pyspark.sql.functions import col, trim

print(f"Rows before cleaning: {df.count()}")

df_email_clean = df.filter(
    ~(
        col("email").isNull() |
        (trim(col("username")).isNull()) |
        (trim(col("username")) == "")
    )
)

print(f"Rows after removing null email OR empty username: {df_email_clean.count()}")
df_email_clean.select("Customer Name", "email", "username").show(10)

Rows before cleaning: 9994
Rows after removing null email OR empty username: 9120
+---------------+--------------------+----------+
|  Customer Name|               email|  username|
+---------------+--------------------+----------+
|    Claire Gute|claire.gute602@gm...|claigut377|
|    Claire Gute|claire.gute276@ou...|claigut631|
| Sean O'Donnell|sean.o'donnell025...|seano'd591|
| Sean O'Donnell|sean.o'donnell016...|seano'd402|
|Brosina Hoffman|brosina.hoffman52...|broshof980|
|Brosina Hoffman|brosina.hoffman80...|broshof511|
|Brosina Hoffman|brosina.hoffman63...|broshof376|
|Brosina Hoffman|brosina.hoffman93...|broshof120|
|Brosina Hoffman|brosina.hoffman08...|broshof270|
|Brosina Hoffman|brosina.hoffman93...| broshof73|
+---------------+--------------------+----------+
only showing top 10 rows


## Q13: How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

**Answer:**

The `.agg()` function accepts multiple aggregate expressions in a single call, computing all of them in **one pass over the data** (one Spark job), rather than running separate `.agg()` calls for each statistic — which would trigger multiple jobs and re-scan the data each time.

Note: The `price` column has 599 null values in this dataset, so we clean it first using `try_cast` (same pattern as `Sales` in Q4) before aggregating.

In [ ]:
from pyspark.sql.functions import min as spark_min, max as spark_max, mean, count, expr, col

# Clean price column first (599 nulls + potential malformed values)
df = df.withColumn("price", expr("try_cast(price as double)"))

price_stats = df.agg(
    spark_min("price").alias("min_price"),
    spark_max("price").alias("max_price"),
    mean("price").alias("mean_price"),
    count("price").alias("non_null_price_count")
)

price_stats.show()

+---------+---------+------------------+--------------------+
|min_price|max_price|        mean_price|non_null_price_count|
+---------+---------+------------------+--------------------+
|    0.444| 22638.48|230.89834953896323|                9110|
+---------+---------+------------------+--------------------+



## Q14: In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

**Answer:**

`inferSchema=True` tells Spark to sample the dataset and automatically guess each column's data type. While convenient, this poses several real risks when data is messy or has inconsistent formats — risks that were directly encountered while working on this assignment:

1. **Type Misdetection from Corrupted Rows** — In this dataset, the `Sales` column was inferred as a numeric type, but 300 rows actually contained text fragments (e.g., `" Black"`, `" Executive Red"`) due to a CSV parsing issue where commas inside the unescaped `Product Name` field shifted column boundaries. When these rows were later processed with strict casting/aggregation (Q4), Spark threw a `CAST_INVALID_INPUT` / `NumberFormatException` and crashed the job entirely — `inferSchema` had silently let bad data into a column it assumed was clean.

2. **Inconsistent Date/Timestamp Formats Cause Parse Failures** — The `raw_timestamp` column in this dataset mixed two formats: `08-11-2016 00:47` (DD-MM-YYYY) and `2016-11-08 00:31:55` (YYYY-MM-DD HH:mm:ss). When converting with the strict `to_timestamp()` function (Q10), Spark threw a `DateTimeException: CANNOT_PARSE_TIMESTAMP` because it expected a single consistent format. `inferSchema` does not detect or warn about this — it typically infers such columns as `StringType`, masking the problem until a downstream operation tries to use the column as a real timestamp.

3. **Sampling-Based Inference Is Not Exhaustive** — By default, `inferSchema` only samples a portion of the data to guess types (not the full dataset for very large files). A column that looks numeric in the sample rows may have non-numeric outliers later in the file — exactly what happened with the `Sales` column here, where the bad rows weren't necessarily in the first few sampled rows.

4. **Expensive Double-Read of Data** — `inferSchema=True` requires Spark to scan the entire file once just to determine types, then read it again to load the data — doubling the read cost. For very large production datasets, this is a meaningful performance penalty, especially since the result (a guessed schema) is unreliable when the source data is messy.

5. **No Error Until Much Later in the Pipeline** — Because schema inference happens silently at read time, problems often surface only later — during an aggregation, a join, or a write — making the root cause (a single malformed source row) much harder to trace back.

**Best Practice:** For production pipelines with known, messy, or evolving data, define an **explicit `StructType` schema** instead of `inferSchema=True`. Combined with `try_cast` / `try_to_timestamp` for safe type conversion (as demonstrated in Q4 and Q10 of this assignment), this surfaces data quality issues as controlled `NULL` values rather than runtime crashes — giving you the chance to inspect and clean bad rows deliberately instead of being blindsided mid-pipeline.

## Q15: Write a final processing pipeline that:
1. Filters out duplicates.
2. Fills null prices with 0.
3. Groups by store_id to calculate total revenue.

**Answer:**

This combines the full Week 5 workflow into one pipeline: deduplicate → clean nulls → aggregate. Total revenue is computed as the sum of `price` per `store_id`.

In [ ]:
from pyspark.sql.functions import sum as spark_sum, round as spark_round, expr, col

# Step 1: Filter out duplicates (based on user_id + transaction_date, as defined in Q3)
df_pipeline = df.dropDuplicates(subset=["user_id", "transaction_date"])
print(f"Rows after deduplication: {df_pipeline.count()}")

# Step 2: Ensure price is numeric, then fill nulls with 0
df_pipeline = df_pipeline.withColumn("price", expr("try_cast(price as double)"))
df_pipeline = df_pipeline.na.fill({"price": 0})

# Step 3: Group by store_id to calculate total revenue
store_revenue = df_pipeline.groupBy("store_id") \
                            .agg(spark_round(spark_sum("price"), 2).alias("total_revenue")) \
                            .orderBy("total_revenue", ascending=False)

store_revenue.show()

Rows after deduplication: 5211
+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|  STR002|    205143.41|
|  STR016|    141784.57|
|  STR006|     73826.01|
|  STR005|     55742.81|
|  STR011|     46240.64|
|  STR010|     44605.21|
|  STR013|     44517.31|
|  STR003|     40920.58|
|  STR018|     38230.58|
|  STR025|     37501.83|
|  STR004|     34597.72|
|  STR031|     23336.13|
|  STR014|     18375.53|
|  STR015|     16994.58|
|  STR001|     16576.81|
|  STR007|     15159.72|
|  STR017|     14366.72|
|  STR033|     13722.17|
|  STR019|     13366.22|
|  STR012|      12558.3|
+--------+-------------+
only showing top 20 rows


In [ ]:
# Save final pipeline result to output/results.csv
store_revenue.toPandas().to_csv("results.csv", index=False)

from google.colab import files
files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>